# Black-box ResNet-50 baseline for CUB-200-2011

This notebook trains a non-interpretable image classifier used only as an accuracy reference for the CBM notebooks.

It follows the same practical structure as the other project notebooks:

- Colab/local path handling;
- standard CUB train/test split;
- ResNet-50 pretrained on ImageNet;
- top-1/top-5/F1/balanced-accuracy evaluation;
- best-checkpoint saving;
- summary CSV export.

Unlike the CBM notebooks, this model has **no concept bottleneck** and therefore should not be used for concept-quality, hierarchy-consistency, or explanation-faithfulness metrics.

In [ ]:
# ============================================================
# 0. Colab/local setup
# ============================================================
from pathlib import Path
import os
import sys
import random
import json
import time
import shutil
import zipfile
import tarfile
import urllib.request

RUN_ON_COLAB = False  # Set True when running on Colab

if RUN_ON_COLAB:
    BASE_DIR = Path('/content')
else:
    BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / 'data'
CUB_DIR = DATA_DIR / 'CUB_200_2011'
SAVE_DIR = BASE_DIR / 'processed'
MODEL_DIR = SAVE_DIR / 'blackbox_resnet50'

DATA_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

NUM_WORKERS = 2 if RUN_ON_COLAB else 0
PIN_MEMORY = True

print('BASE_DIR:', BASE_DIR)
print('DATA_DIR:', DATA_DIR)
print('CUB_DIR:', CUB_DIR)
print('SAVE_DIR:', SAVE_DIR)
print('NUM_WORKERS:', NUM_WORKERS)

In [ ]:
# Optional installs for Colab. Local environments are assumed to already have these.
if RUN_ON_COLAB:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torchmetrics'])

In [ ]:
# ============================================================
# 1. Imports and reproducibility
# ============================================================
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from torchvision import models

from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
)

import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 2. Dataset availability

Local mode expects the CUB folder at:

```text
./data/CUB_200_2011
```

Colab mode can optionally download/extract the dataset if you provide a valid archive URL. The automatic download is deliberately conservative because the official CUB links may change or be blocked.

In [ ]:
# ============================================================
# 2. Optional dataset download/extraction
# ============================================================
AUTO_DOWNLOAD_IN_COLAB = False
CUB_ARCHIVE_URL = ''  # Optional: set to a working CUB archive URL if needed.


def extract_archive(archive_path: Path, dest_dir: Path):
    archive_path = Path(archive_path)
    dest_dir = Path(dest_dir)
    print(f'Extracting {archive_path} -> {dest_dir}')
    if archive_path.suffix == '.zip':
        with zipfile.ZipFile(archive_path, 'r') as zf:
            zf.extractall(dest_dir)
    elif archive_path.suffixes[-2:] == ['.tar', '.gz'] or archive_path.suffix == '.tgz':
        with tarfile.open(archive_path, 'r:gz') as tf:
            tf.extractall(dest_dir)
    else:
        raise ValueError(f'Unsupported archive type: {archive_path}')

if RUN_ON_COLAB and AUTO_DOWNLOAD_IN_COLAB and not CUB_DIR.exists():
    if not CUB_ARCHIVE_URL:
        raise ValueError('Set CUB_ARCHIVE_URL before enabling AUTO_DOWNLOAD_IN_COLAB.')
    archive_path = DATA_DIR / Path(CUB_ARCHIVE_URL).name
    if not archive_path.exists():
        print('Downloading CUB archive...')
        urllib.request.urlretrieve(CUB_ARCHIVE_URL, archive_path)
    extract_archive(archive_path, DATA_DIR)

required_files = [
    CUB_DIR / 'images.txt',
    CUB_DIR / 'image_class_labels.txt',
    CUB_DIR / 'train_test_split.txt',
    CUB_DIR / 'classes.txt',
    CUB_DIR / 'images',
]
missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError(
        'CUB dataset not found or incomplete'
    )
print('CUB dataset found.')

In [ ]:
# ============================================================
# 3. Metadata loading
# ============================================================
def load_txt(filename, col_names, sep=' '):
    return pd.read_csv(CUB_DIR / filename, sep=sep, header=None, names=col_names)

images_df  = load_txt('images.txt',             ['img_id', 'filepath'])
labels_df  = load_txt('image_class_labels.txt', ['img_id', 'class_id'])
split_df   = load_txt('train_test_split.txt',   ['img_id', 'is_train'])
classes_df = load_txt('classes.txt',            ['class_id', 'class_name'])

master_df = images_df.merge(labels_df, on='img_id').merge(split_df, on='img_id')
master_df['label'] = master_df['class_id'] - 1
master_df['image_path'] = master_df['filepath'].apply(lambda p: CUB_DIR / 'images' / p)

N_CLASSES = int(master_df['label'].nunique())
print(master_df.head())
print('N images:', len(master_df))
print('N classes:', N_CLASSES)
print('Train images:', int(master_df['is_train'].sum()))
print('Test images:', int((1 - master_df['is_train']).sum()))

# Sanity check image paths
bad_paths = master_df.loc[~master_df['image_path'].apply(lambda p: Path(p).exists())]
if len(bad_paths) > 0:
    raise FileNotFoundError(f'{len(bad_paths)} image paths are missing. Example: {bad_paths.iloc[0]["image_path"]}')

In [ ]:
# ============================================================
# 4. Dataset and dataloaders
# ============================================================
IMG_SIZE = 224
BATCH_SIZE = 64

train_transform = T.Compose([
    T.Resize(256),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class CUBImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = Path(row['image_path'])
        img = Image.open(path).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        label = int(row['label'])
        return img, label

train_df = master_df[master_df['is_train'] == 1].copy()
test_df  = master_df[master_df['is_train'] == 0].copy()

train_dataset = CUBImageDataset(train_df, transform=train_transform)
test_dataset  = CUBImageDataset(test_df,  transform=test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY and torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY and torch.cuda.is_available(),
)

batch = next(iter(train_loader))
print('Batch image shape:', batch[0].shape)
print('Batch label shape:', batch[1].shape)

In [ ]:
# ============================================================
# 5. Black-box ResNet-50 model
# ============================================================
class BlackBoxResNet50(nn.Module):
    def __init__(self, n_classes=200, pretrained=True):
        super().__init__()
        if pretrained:
            weights = models.ResNet50_Weights.IMAGENET1K_V2
        else:
            weights = None
        self.backbone = models.resnet50(weights=weights)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, n_classes)

    def forward(self, x):
        return self.backbone(x)

model = BlackBoxResNet50(n_classes=N_CLASSES, pretrained=True).to(device)
print(model.backbone.fc)

In [ ]:
# ============================================================
# 6. Optimizer, scheduler, losses
# ============================================================
EPOCHS = 40
LR_BACKBONE = 1e-4
LR_HEAD = 1e-3
WEIGHT_DECAY = 1e-4

# Use a smaller LR for pretrained layers and a larger LR for the final head.
head_params = list(model.backbone.fc.parameters())
head_param_ids = {id(p) for p in head_params}
backbone_params = [p for p in model.parameters() if id(p) not in head_param_ids]

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': LR_BACKBONE},
    {'params': head_params, 'lr': LR_HEAD},
], weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
)

criterion = nn.CrossEntropyLoss()

In [ ]:
# ============================================================
# 7. Evaluation utilities
# ============================================================
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    total = 0
    correct1 = 0
    correct5 = 0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, desc='eval', leave=False):
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)
        n = images.size(0)

        total_loss += loss.item() * n
        total += n

        top5 = logits.topk(5, dim=1).indices
        pred1 = top5[:, 0]
        correct1 += (pred1 == labels).sum().item()
        correct5 += (top5 == labels.unsqueeze(1)).any(dim=1).sum().item()

        all_preds.extend(pred1.detach().cpu().tolist())
        all_labels.extend(labels.detach().cpu().tolist())

    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    bal_acc = balanced_accuracy_score(all_labels, all_preds)

    return {
        'loss': total_loss / total,
        'top1_accuracy': correct1 / total,
        'top5_accuracy': correct5 / total,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'balanced_accuracy': bal_acc,
        'preds': all_preds,
        'labels': all_labels,
    }


def print_metrics(prefix, metrics):
    print(
        f"{prefix} | "
        f"loss={metrics['loss']:.4f} | "
        f"top1={metrics['top1_accuracy']:.4f} | "
        f"top5={metrics['top5_accuracy']:.4f} | "
        f"macroF1={metrics['macro_f1']:.4f} | "
        f"balAcc={metrics['balanced_accuracy']:.4f}"
    )

In [ ]:
# ============================================================
# 8. Training loop
# ============================================================
BEST_CKPT = MODEL_DIR / 'blackbox_resnet50_best.pt'
LAST_CKPT = MODEL_DIR / 'blackbox_resnet50_last.pt'
HISTORY_CSV = MODEL_DIR / 'blackbox_resnet50_history.csv'

best_top1 = -1.0
history = []

start_time = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    total = 0
    correct = 0

    pbar = tqdm(train_loader, desc=f'epoch {epoch:02d}/{EPOCHS}', leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        n = images.size(0)
        train_loss += loss.item() * n
        total += n
        pred = logits.argmax(dim=1)
        correct += (pred == labels).sum().item()
        pbar.set_postfix(loss=loss.item(), acc=correct / total)

    train_loss /= total
    train_acc = correct / total

    test_metrics = evaluate(model, test_loader, device)
    scheduler.step(test_metrics['top1_accuracy'])

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'train_acc': train_acc,
        **{f'test_{k}': v for k, v in test_metrics.items() if k not in ['preds', 'labels']},
        'lr_backbone': optimizer.param_groups[0]['lr'],
        'lr_head': optimizer.param_groups[1]['lr'],
    }
    history.append(row)
    pd.DataFrame(history).to_csv(HISTORY_CSV, index=False)

    print(
        f"Epoch {epoch:02d}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
        f"test_top1={test_metrics['top1_accuracy']:.4f}, test_top5={test_metrics['top5_accuracy']:.4f}, "
        f"macroF1={test_metrics['macro_f1']:.4f}"
    )

    if test_metrics['top1_accuracy'] > best_top1:
        best_top1 = test_metrics['top1_accuracy']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics': test_metrics,
            'config': {
                'model': 'BlackBoxResNet50',
                'n_classes': N_CLASSES,
                'epochs': EPOCHS,
                'lr_backbone': LR_BACKBONE,
                'lr_head': LR_HEAD,
                'weight_decay': WEIGHT_DECAY,
                'img_size': IMG_SIZE,
                'batch_size': BATCH_SIZE,
                'seed': SEED,
            },
        }, BEST_CKPT)
        print(f'  saved best checkpoint: {BEST_CKPT}')

torch.save({
    'epoch': EPOCHS,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'history': history,
}, LAST_CKPT)

elapsed = time.time() - start_time
print(f'Training complete. Best top-1={best_top1:.4f}. Elapsed={elapsed/60:.1f} min')

In [ ]:
# ============================================================
# 9. Load best checkpoint strictly and compute final metrics
# ============================================================
ckpt = torch.load(BEST_CKPT, map_location=device)
missing, unexpected = model.load_state_dict(ckpt['model_state_dict'], strict=False)
if missing or unexpected:
    print('Missing keys:', missing)
    print('Unexpected keys:', unexpected)
    raise RuntimeError('Checkpoint does not exactly match the black-box model definition.')

final_metrics = evaluate(model, test_loader, device)
print_metrics('Final black-box ResNet-50', final_metrics)

summary = pd.DataFrame([{
    'model': 'Black-box ResNet-50',
    'top1_accuracy': final_metrics['top1_accuracy'],
    'top5_accuracy': final_metrics['top5_accuracy'],
    'macro_f1': final_metrics['macro_f1'],
    'weighted_f1': final_metrics['weighted_f1'],
    'balanced_accuracy': final_metrics['balanced_accuracy'],
    'concept_quality': 'N/A',
    'hierarchy_consistency': 'N/A',
    'explainability_role': 'Non-interpretable accuracy reference only',
}])

summary_path = MODEL_DIR / 'blackbox_resnet50_summary.csv'
summary.to_csv(summary_path, index=False)
print('Saved:', summary_path)
summary

In [ ]:
# ============================================================
# 10. Per-class accuracy and confusion matrix export
# ============================================================
y_true = np.array(final_metrics['labels'])
y_pred = np.array(final_metrics['preds'])

per_class_rows = []
for cls_idx in range(N_CLASSES):
    mask = y_true == cls_idx
    support = int(mask.sum())
    acc = float((y_pred[mask] == cls_idx).mean()) if support > 0 else np.nan
    class_name = classes_df.loc[classes_df['class_id'] == cls_idx + 1, 'class_name'].values[0]
    per_class_rows.append({
        'class_id': cls_idx + 1,
        'label': cls_idx,
        'class_name': class_name,
        'support': support,
        'accuracy': acc,
    })

per_class_df = pd.DataFrame(per_class_rows)
per_class_path = MODEL_DIR / 'blackbox_resnet50_per_class_accuracy.csv'
per_class_df.to_csv(per_class_path, index=False)
print('Saved:', per_class_path)

cm = confusion_matrix(y_true, y_pred, labels=list(range(N_CLASSES)))
cm_path = MODEL_DIR / 'blackbox_resnet50_confusion_matrix.npy'
np.save(cm_path, cm)
print('Saved:', cm_path)

per_class_df.sort_values('accuracy').head(10)

In [ ]:
# ============================================================
# 11. Training curve
# ============================================================
history_df = pd.read_csv(HISTORY_CSV)

plt.figure(figsize=(7, 4))
plt.plot(history_df['epoch'], history_df['train_acc'], label='Train top-1')
plt.plot(history_df['epoch'], history_df['test_top1_accuracy'], label='Test top-1')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Black-box ResNet-50 training curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
fig_path = MODEL_DIR / 'blackbox_resnet50_training_curve.png'
plt.savefig(fig_path, dpi=150)
plt.show()
print('Saved:', fig_path)

## How to use this result in the report

This model is **not explainable-by-design**. It should only be included in the predictive-performance table as a non-interpretable accuracy reference.

Do not include it in:

- concept quality tables;
- hierarchy consistency tables;
- concept intervention analysis;
- concept relevance analysis;
- saliency-based concept grounding discussion.

Suggested wording:

> The black-box ResNet-50 provides an upper-bound-style accuracy reference. The gap between this model and the CBMs estimates the cost of enforcing an interpretable concept bottleneck.